In [1]:
import pandas as pd

# ============================================================
# Step 0: Load all 7 cleaned tables
# ============================================================
df_demography = pd.read_csv("Team05_PyCraft_Cleaned_data/demography_cleaned.csv")
df_hospitalization = pd.read_csv("Team05_PyCraft_Cleaned_data/hospitalization_discharge_cleaned.csv")
df_patienthistory = pd.read_csv("Team05_PyCraft_Cleaned_data/patienthistory_cleaned.csv")
df_cardiac = pd.read_csv("Team05_PyCraft_Cleaned_data/cardiac_cleaned.csv")
df_labs = pd.read_csv("Team05_PyCraft_Cleaned_data/labs_cleaned.csv")
df_responsivenes = pd.read_csv("Team05_PyCraft_Cleaned_data/responsivenes_cleaned.csv")
df_precriptions = pd.read_csv("Team05_PyCraft_Cleaned_data/precriptions_cleaned.csv")

print("Starting patient count (demography):", df_demography['patient_id'].nunique())

# ============================================================
# Step 1: Merge all one-to-one tables onto demography
# ============================================================
wide = df_demography.merge(df_hospitalization, on='patient_id', how='left')
wide = wide.merge(df_patienthistory, on='patient_id', how='left')
wide = wide.merge(df_responsivenes, on='patient_id', how='left')
wide = wide.merge(df_cardiac, on='patient_id', how='left')
wide = wide.merge(df_labs, on='patient_id', how='left')

print("Shape after one-to-one merges:", wide.shape)
print("Duplicate patient_ids after merge:", wide['patient_id'].duplicated().sum())

# ============================================================
# Step 2: Aggregate prescriptions (many-to-one) and merge
# ============================================================
# One-hot flag per drug: did the patient receive it?
drug_flags = pd.crosstab(df_precriptions['patient_id'], df_precriptions['drug_name'])
drug_flags = (drug_flags > 0).astype(int)
drug_flags.columns = [f"drug_{c.strip().replace(' ', '_')}" for c in drug_flags.columns]
drug_flags = drug_flags.reset_index()

# Count of distinct drugs per patient
drug_count = (
    df_precriptions.groupby('patient_id')['drug_name']
    .nunique()
    .rename('num_distinct_drugs')
    .reset_index()
)

wide = wide.merge(drug_flags, on='patient_id', how='left')
wide = wide.merge(drug_count, on='patient_id', how='left')

# Patients with no prescriptions on record → genuinely 0 drugs, not unknown
drug_cols = [c for c in wide.columns if c.startswith('drug_')] + ['num_distinct_drugs']
wide[drug_cols] = wide[drug_cols].fillna(0)

print("Shape after adding prescriptions:", wide.shape)

# ============================================================
# Step 3: Validate the final wide table
# ============================================================
print("Final wide table shape:", wide.shape)
print("Duplicate patient_ids:", wide['patient_id'].duplicated().sum())
print("Total columns:", wide.shape[1])

# Quick look at which columns have the most missing data (informs imputation later)
print(wide.isna().sum().sort_values(ascending=False).head(15))

# ============================================================
# Step 4: Save the wide table
# ============================================================
wide.to_csv("Team05_PyCraft_Cleaned_data/wide_merged_table.csv", index=False)
print("Saved to cleaned/wide_merged_table.csv — shape:", wide.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'Team05_PyCraft_Cleaned_data/demography_cleaned.csv'